In [ ]:
use role sysadmin;
use warehouse compute_wh;
use schema legacy_db.consumption_sch;

****
### Customer DIM table
****

In [ ]:
create or replace table legacy_db.consumption_sch.customer_dim (
c_dim_id int primary key autoincrement,
cust_key number,
name text,
address text,
nation_name text,
phone text,
acct_bal number,
mkt_segment text
);

****
### Date DIM table
****

In [ ]:
create or replace table legacy_db.consumption_sch.date_dim (
d_dim_id int primary key autoincrement,
order_dt date,
order_year number(4) default -1,
order_quarter number(1) default -1,
order_month number(2) default -1,
order_week number(2) default -1,
order_day number(2) default -1
);

****
### Priority DIM table
****

In [ ]:
create or replace table legacy_db.consumption_sch.priority_dim (
p_dim_id int primary key autoincrement,
order_priority text
);

****
### Order FACT table
****

In [ ]:
create or replace table legacy_db.consumption_sch.order_fact (
o_fact_id int primary key autoincrement,
d_dim_id_fk number,
c_dim_id_fk number,
p_dim_id_fk number,
order_key number,
total_price number
);

****
### Tasks to Populate tables using merge statements
****

In [ ]:
create or replace task legacy_db.raw_sch.populate_customer_dim_task
    warehouse = compute_wh
    after legacy_db.raw_sch.populate_clean_customer_task
    when system$stream_has_data('legacy_db.clean_sch.customer_clean_stream')
        as
    merge into legacy_db.consumption_sch.customer_dim as to_dim 
    using (
            select 
                cust_key,
                name,
                address,
                nation_name,
                phone,
                acct_bal,
                mkt_segment
            from legacy_db.clean_sch.customer_clean_stream           
        ) as from_clean on
        to_dim.cust_key = from_clean.cust_key
        when matched
        then update set
            to_dim.name = from_clean.name,
            to_dim.address = from_clean.address,
            to_dim.nation_name = from_clean.nation_name,
            to_dim.phone = from_clean.phone,
            to_dim.acct_bal = from_clean.acct_bal,
            to_dim.mkt_segment = from_clean.mkt_segment
        when not matched
        then insert (cust_key,name,address,nation_name,phone,acct_bal,mkt_segment)
        values (
            from_clean.name,
            from_clean.address,
            from_clean.nation_name,
            from_clean.phone,
            from_clean.acct_bal,
            from_clean.mkt_segment
        );

In [ ]:
create or replace task legacy_db.raw_sch.populate_date_dim_task
    warehouse = compute_wh
    after legacy_db.raw_sch.populate_clean_order_task
    when system$stream_has_data('legacy_db.clean_sch.order_clean_stream_for_dt')
        as 
    merge into legacy_db.consumption_sch.date_dim as to_dim 
    using (
            select order_date
            from legacy_db.clean_sch.order_clean_stream_for_dt
            group by order_date
    ) as from_clean
    on to_dim.order_dt = from_clean.order_date
    when not matched 
    then insert (order_dt,order_year,order_quarter,order_month,order_week,order_day )
    values 
    (
        from_clean.order_date,
        year(from_clean.order_date),
        quarter(from_clean.order_date),
        month(from_clean.order_date),
        week(from_clean.order_date),
        dayofmonth(from_clean.order_date)
    );

In [ ]:
create or replace task legacy_db.raw_sch.populate_priority_dim_task
    warehouse = compute_wh
    after legacy_db.raw_sch.populate_clean_order_task
    when system$stream_has_data('legacy_db.clean_Sch.order_clean_stream_for_priority')
        as
    merge into legacy_db.consumption_sch.priority_dim as to_dim 
    using (
            select 
                order_priority
            from legacy_db.clean_Sch.order_clean_stream_for_priority
            group by order_priority
    ) as from_clean on
    to_dim.order_priority = from_clean.order_priority
    when not matched 
    then insert (order_priority)
    values (
                from_clean.order_priority
    );

In [ ]:
create or replace task legacy_db.raw_sch.populate_order_fact_task
warehouse = compute_wh
after legacy_db.raw_sch.populate_customer_dim_task,legacy_db.raw_sch.populate_date_dim_task,legacy_db.raw_sch.populate_priority_dim_task
    when system$stream_has_data('legacy_db.clean_sch.order_clean_stream')
        as 
    merge into legacy_db.consumption_sch.order_fact as to_fact
    using (
            select 
                cd.c_dim_id,
                dd.d_dim_id,
                pd.p_dim_id,
                oc.order_key,
                oc.total_price
            from 
            legacy_db.clean_sch.order_clean_stream oc 
            join legacy_db.consumption_sch.customer_dim cd on cd.cust_key = oc.cust_key
            join legacy_db.consumption_sch.date_dim dd on dd.order_dt = oc.order_date
            join legacy_db.consumption_sch.priority_dim pd on pd.order_priority = oc.order_priority
     ) as from_clean on
     to_fact.order_key = from_clean.order_key
     when not matched 
     then insert (d_dim_id_fk,c_dim_id_fk,p_dim_id_fk,order_key,total_price)
    values 
    (
        from_clean.c_dim_id ,
        from_clean.d_dim_id ,
        from_clean.p_dim_id ,
        from_clean.order_key ,
        from_clean.total_price
    );